# Project: Wildfire Mapping

Goal: Build an html side with an interactive map where the user can see the recents wildfires. Provide the user with information of phisical size, duration, intensity, etc. of the wildfires. Use pop-ups and tooltips to make the map interactive and structured for the users. The map should be for public users which are interesting in wildfires.

Tasks:
1. Load the api
2. Extract the data which is used to locate the wildfire (VIIRS_SNPP_NRT)
3. Explore the data
4. Clean the data (if needed)
5. Check wildfires for different properties
6. Visulaization of the wildfires
7. Provide additional information about the wildfires. 
8. Make the map interactive

In [115]:
# import all libraries
import requests
import pandas as pd
import io
import geopandas as gpd
import folium
from folium.plugins import MarkerCluster
from datetime import datetime
from shapely.geometry import MultiPoint
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import json
import numpy as np

In [116]:
# Building the api url 

# get api key
api_key = "47778cf594ae276d2b7dfc098596de2a"
# define source
api_source = "VIIRS_SNPP_NRT" # or change ot to VIIRS_SNPP_SP?
# define area coordinates
api_area_coordinates = "world"
# day range. Days going back from today
api_day_range = 5
# build api url with api key
api_url = f"https://firms.modaps.eosdis.nasa.gov/api/area/csv/{api_key}/{api_source}/{api_area_coordinates}/{api_day_range}"

In [117]:
# load the data form the api
response = requests.get(api_url)

# check if api import was successfull
if response.status_code == 200:
    print("API request successfull")

    # get the data as a csv
    data_csv = response.text
    # create a dataframe
    data_df = pd.read_csv(io.StringIO(data_csv))

else:
    print(f"Request failed. Status code: {response.status_code}")

API request successfull


In [135]:
# converting the data data frame into a geo-data frame
wildfire_gdf_crs4326 = gpd.GeoDataFrame(data_df, geometry=gpd.points_from_xy(data_df["longitude"], data_df["latitude"]), crs=4326)

# convert the date column into a date datetime type
wildfire_gdf_crs4326["acq_date"] = pd.to_datetime(wildfire_gdf_crs4326["acq_date"], format="%Y-%m-%d")

# add a column with the date
wildfire_gdf_crs4326["acq_date_day"] = wildfire_gdf_crs4326["acq_date"].dt.date

# preject to calculate in meters
wildfire_gdf_crs3857 = wildfire_gdf_crs4326.to_crs(epsg=3857)
# join nearby spatial points according to the same fire (most likly the same)
wildfire_gdf_crs3857["geometry_buffer"] = wildfire_gdf_crs3857.geometry.buffer(1501) # buffer of 1501 meters. resolution of the data is 350mx750m
# join fires within the buffer
joined_wildfires = gpd.sjoin(
    wildfire_gdf_crs4326[["acq_date", "geometry"]],
    wildfire_gdf_crs3857[["acq_date", "geometry_buffer"]].set_geometry("geometry_buffer"),
    how="left", # make sure to keep all points
    predicate="within"
)

joined_wildfires = joined_wildfires[joined_wildfires["acq_date_left"] == joined_wildfires["acq_date_right"]] # Keep only fires with the same registration date
# aggregate into clusters
wildfire_clusters = (
    joined_wildfires.groupby("index_right").agg(
        acq_date=("acq_date_left", "first"),
        count=("acq_date_left", "count"),
        geometry=("geometry", lambda geoms: MultiPoint(list(geoms)).centroid)
    )
    .reset_index(drop=True)
)

# Build the cleaned aggregated wildfire data
wildfire_clusterd = gpd.GeoDataFrame(wildfire_clusters, geometry="geometry", crs=3857).to_crs(epsg=4326)


C:\Users\lilsh\AppData\Local\Temp\ipykernel_21536\2379037073.py:15: UserWarning: CRS mismatch between the CRS of left geometries and the CRS of right geometries.
Use `to_crs()` to reproject one of the input geometries to match the CRS of the other.

Left CRS: EPSG:4326
Right CRS: EPSG:3857

  joined_wildfires = gpd.sjoin(


In [136]:
# verify transformation to geo data frame
display(wildfire_gdf_crs4326.head(5))
display(wildfire_clusterd.head(5))
display(wildfire_clusterd.info)
display(wildfire_clusterd.dtypes)

,latitude,longitude,bright_ti4,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_ti5,frp,daynight,geometry,acq_date_day
0,22.70248,39.04295,307.43,0.50,0.66,2026-05-12,1,N,VIIRS,n,2.0NRT,292.96,1.04,N,POINT (39.04295 22.70248),2026-05-12
1,23.94810,38.26667,325.12,0.42,0.61,2026-05-12,1,N,VIIRS,n,2.0NRT,295.73,2.29,N,POINT (38.26667 23.9481),2026-05-12
2,23.95083,38.26488,327.78,0.42,0.61,2026-05-12,1,N,VIIRS,n,2.0NRT,295.62,2.88,N,POINT (38.26488 23.95083),2026-05-12
3,24.27637,37.56365,307.05,0.38,0.58,2026-05-12,1,N,VIIRS,n,2.0NRT,295.67,0.95,N,POINT (37.56365 24.27637),2026-05-12
4,24.95234,32.91698,337.41,0.39,0.44,2026-05-12,1,N,VIIRS,n,2.0NRT,292.75,4.46,N,POINT (32.91698 24.95234),2026-05-12


,acq_date,count,geometry


<bound method DataFrame.info of Empty GeoDataFrame
Columns: [acq_date, count, geometry]
Index: []>

acq_date    datetime64[us]
count                int64
geometry          geometry
dtype: object

In [7]:
# initalize Folium back ground map
background_map = folium.Map(
    location=[0, 0], # start zoom at latitude and longitude 0
    zoom_start=2, # shows the whole word at the start
    tiles="CartoDB DarkMatter", # Dark basmap
    control_scale=True # Add scalebar
)

cluster_fire = MarkerCluster(name="Recent Fires").add_to(background_map)
# build markers for the wildfires
for idx, row in wildfire_clusterd.iterrows():
    lat = row.geometry.y # extract latitude out of the geometry column 
    lon = row.geometry.x # eextract longitude out of the geometry column
    count = row["count"] # counting how many fires are aggregated

    # Formating the Tooltip. Date of the fire registation
    fire_start = row["acq_date"].date()
    tooltip = f"Date: {fire_start} | Detections: {count}"

    # Define Marker color deoending the count of fires
    if count == 1:
        color =  "orange"
    elif count <= 5:
        color = "red"
    else:
        color = "darkred"

    # create Marker of the fire locations
    folium.Marker(
        location=[lat, lon],
        tooltip=tooltip,
        icon=folium.Icon(color=color, icon="fire", prefix="fa")
    ).add_to(cluster_fire)

folium.LayerControl().add_to(background_map)
background_map.save("../outputs/map.html")

In addition map I also decided to give some Insighs about in which region most open fires are recorded

In [85]:
# load Geo Data frame of the world countries
world_import = gpd.read_file("https://naturalearth.s3.amazonaws.com/10m_cultural/ne_10m_admin_0_countries.zip").to_crs(epsg=4326) # load Countries from online source.

# inspect worl gdf
display(world_import.head(5))
print(world_import.columns.tolist())


# cleaning the world 
world = world_import[["NAME", "ISO_A3", "CONTINENT", "SUBREGION", "geometry"]] # just keep usefull attributes ot of the world gdf.
world["AREA_KM2"] = world.to_crs(epsg=3857).geometry.area / 1e6 # calulate the area
display(world)

,featurecla,scalerank,LABELRANK,SOVEREIGNT,SOV_A3,ADM0_DIF,LEVEL,TYPE,TLC,ADMIN,...,FCLASS_TR,FCLASS_ID,FCLASS_PL,FCLASS_GR,FCLASS_IT,FCLASS_NL,FCLASS_SE,FCLASS_BD,FCLASS_UA,geometry
0,Admin-0 country,0,2,Indonesia,IDN,0,2,Sovereign country,1,Indonesia,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"MULTIPOLYGON (((117.70361 4.16341, 117.70361 4..."
1,Admin-0 country,0,3,Malaysia,MYS,0,2,Sovereign country,1,Malaysia,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"MULTIPOLYGON (((117.70361 4.16341, 117.69711 4..."
2,Admin-0 country,0,2,Chile,CHL,0,2,Sovereign country,1,Chile,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"MULTIPOLYGON (((-69.51009 -17.50659, -69.50611..."
3,Admin-0 country,0,3,Bolivia,BOL,0,2,Sovereign country,1,Bolivia,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"POLYGON ((-69.51009 -17.50659, -69.51009 -17.5..."
4,Admin-0 country,0,2,Peru,PER,0,2,Sovereign country,1,Peru,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"MULTIPOLYGON (((-69.51009 -17.50659, -69.63832..."


['featurecla', 'scalerank', 'LABELRANK', 'SOVEREIGNT', 'SOV_A3', 'ADM0_DIF', 'LEVEL', 'TYPE', 'TLC', 'ADMIN', 'ADM0_A3', 'GEOU_DIF', 'GEOUNIT', 'GU_A3', 'SU_DIF', 'SUBUNIT', 'SU_A3', 'BRK_DIFF', 'NAME', 'NAME_LONG', 'BRK_A3', 'BRK_NAME', 'BRK_GROUP', 'ABBREV', 'POSTAL', 'FORMAL_EN', 'FORMAL_FR', 'NAME_CIAWF', 'NOTE_ADM0', 'NOTE_BRK', 'NAME_SORT', 'NAME_ALT', 'MAPCOLOR7', 'MAPCOLOR8', 'MAPCOLOR9', 'MAPCOLOR13', 'POP_EST', 'POP_RANK', 'POP_YEAR', 'GDP_MD', 'GDP_YEAR', 'ECONOMY', 'INCOME_GRP', 'FIPS_10', 'ISO_A2', 'ISO_A2_EH', 'ISO_A3', 'ISO_A3_EH', 'ISO_N3', 'ISO_N3_EH', 'UN_A3', 'WB_A2', 'WB_A3', 'WOE_ID', 'WOE_ID_EH', 'WOE_NOTE', 'ADM0_ISO', 'ADM0_DIFF', 'ADM0_TLC', 'ADM0_A3_US', 'ADM0_A3_FR', 'ADM0_A3_RU', 'ADM0_A3_ES', 'ADM0_A3_CN', 'ADM0_A3_TW', 'ADM0_A3_IN', 'ADM0_A3_NP', 'ADM0_A3_PK', 'ADM0_A3_DE', 'ADM0_A3_GB', 'ADM0_A3_BR', 'ADM0_A3_IL', 'ADM0_A3_PS', 'ADM0_A3_SA', 'ADM0_A3_EG', 'ADM0_A3_MA', 'ADM0_A3_PT', 'ADM0_A3_AR', 'ADM0_A3_JP', 'ADM0_A3_KO', 'ADM0_A3_VN', 'ADM0_A3_TR', 'AD

,NAME,ISO_A3,CONTINENT,SUBREGION,geometry,AREA_KM2
0,Indonesia,IDN,Asia,South-Eastern Asia,"MULTIPOLYGON (((117.70361 4.16341, 117.70361 4...",1.901567e+06
1,Malaysia,MYS,Asia,South-Eastern Asia,"MULTIPOLYGON (((117.70361 4.16341, 117.69711 4...",3.317439e+05
2,Chile,CHL,South America,South America,"MULTIPOLYGON (((-69.51009 -17.50659, -69.50611...",1.255936e+06
3,Bolivia,BOL,South America,South America,"POLYGON ((-69.51009 -17.50659, -69.51009 -17.5...",1.194826e+06
4,Peru,PER,South America,South America,"MULTIPOLYGON (((-69.51009 -17.50659, -69.63832...",1.339975e+06
...,...,...,...,...,...,...
253,Macao,MAC,Asia,Eastern Asia,"MULTIPOLYGON (((113.5586 22.16303, 113.56943 2...",3.523246e+01
254,Ashmore and Cartier Is.,-99,Oceania,Australia and New Zealand,"POLYGON ((123.59702 -12.42832, 123.59775 -12.4...",2.843784e+00
255,Bajo Nuevo Bank,-99,North America,Caribbean,"POLYGON ((-79.98929 15.79495, -79.98782 15.796...",3.895604e-02
256,Serranilla Bank,-99,North America,Caribbean,"POLYGON ((-78.63707 15.86209, -78.64041 15.864...",1.144447e-01


In [86]:
# calculate fires in the diffrent countries
# here the wildfire cluster is used, so nearby fires are counted as one fire and not as several

wildfire_countries = gpd.sjoin(wildfire_clusterd, world, 
                                how="left", predicate="within") # assign to each fire in which country it is located

# Check for fires not located inside a country for example this are burning ships, fires in artica or antarctica
unmatched = wildfire_countries[wildfire_countries["NAME"].isna()]
print(f"Unmatched fires: {len(unmatched)}")
display(wildfire_countries.head(5))

Unmatched fires: 1539


,acq_date,count,geometry,index_right,NAME,ISO_A3,CONTINENT,SUBREGION,AREA_KM2
0,2026-05-12,1,POINT (39.04295 22.70248),105.0,Saudi Arabia,SAU,Asia,Western Asia,2.328558e+06
1,2026-05-12,3,POINT (38.26604 23.94927),105.0,Saudi Arabia,SAU,Asia,Western Asia,2.328558e+06
2,2026-05-12,3,POINT (38.26604 23.94927),105.0,Saudi Arabia,SAU,Asia,Western Asia,2.328558e+06
3,2026-05-12,2,POINT (37.56424 24.276),105.0,Saudi Arabia,SAU,Asia,Western Asia,2.328558e+06
4,2026-05-12,2,POINT (32.91509 24.95271),161.0,Egypt,EGY,Africa,Northern Africa,1.258412e+06


In [88]:
# count the fires for each country
fire_per_country = world[["NAME", "CONTINENT","ISO_A3", "geometry", "AREA_KM2"]].copy() # copy world to build fire per country
# merge fire counts in — countries with no fires get NaN
fire_per_country = fire_per_country.merge(
    wildfire_countries.groupby("NAME").size().reset_index(name="fire_count"),
    on="NAME",
    how="left"  # keep all countries from world
)

fire_per_country["fire_count"] = fire_per_country["fire_count"].fillna(0).astype(int) # fill the NaN values of the countries without fires with 0

fire_per_country = gpd.GeoDataFrame(fire_per_country, geometry="geometry", crs=4326) # 

display(fire_per_country)

,NAME,CONTINENT,ISO_A3,geometry,AREA_KM2,fire_count
0,Indonesia,Asia,IDN,"MULTIPOLYGON (((117.70361 4.16341, 117.70361 4...",1.901567e+06,505
1,Malaysia,Asia,MYS,"MULTIPOLYGON (((117.70361 4.16341, 117.69711 4...",3.317439e+05,74
2,Chile,South America,CHL,"MULTIPOLYGON (((-69.51009 -17.50659, -69.50611...",1.255936e+06,527
3,Bolivia,South America,BOL,"POLYGON ((-69.51009 -17.50659, -69.51009 -17.5...",1.194826e+06,325
4,Peru,South America,PER,"MULTIPOLYGON (((-69.51009 -17.50659, -69.63832...",1.339975e+06,301
...,...,...,...,...,...,...
253,Macao,Asia,MAC,"MULTIPOLYGON (((113.5586 22.16303, 113.56943 2...",3.523246e+01,0
254,Ashmore and Cartier Is.,Oceania,-99,"POLYGON ((123.59702 -12.42832, 123.59775 -12.4...",2.843784e+00,0
255,Bajo Nuevo Bank,North America,-99,"POLYGON ((-79.98929 15.79495, -79.98782 15.796...",3.895604e-02,0
256,Serranilla Bank,North America,-99,"POLYGON ((-78.63707 15.86209, -78.64041 15.864...",1.144447e-01,0


In [109]:
# make a map displaying the nubers of fire per each country
geojson = json.loads(fire_per_country.to_json())


max_fires = fire_per_country["fire_count"].max()
# custom colorscale: grey for 0, then OrRd for fires
colorscale = [
    [0, "lightgrey"],
    [0.0001, "lightyellow"],
    [0.5, "orange"],
    [1, "darkred"]
]

fig = go.Figure(go.Choropleth(
    geojson=geojson,
    locations=fire_per_country["NAME"],
    featureidkey="properties.NAME",
    z=fire_per_country["fire_count"],
    colorscale=colorscale,
    colorbar_title="Fires recorded",
    customdata=fire_per_country[["NAME", "fire_count"]].values,
    hovertemplate="<b>%{customdata[0]}</b><br>Fires recorded: %{customdata[1]}<extra></extra>",
))

fig.update_layout(
    title=dict(text="Fires per Country", x=0.5, xanchor="center", font=dict(size=24)),
    geo=dict(
        showframe=True,
        framecolor="grey",
        showland=True,
        landcolor="lightblue", # same color as ocean
        showcoastlines=False,
        showocean=True,
        oceancolor="lightblue",
        showlakes=True,
        lakecolor="lightblue",
        lataxis=dict(showgrid=True, gridcolor="grey", dtick=30),
        lonaxis=dict(showgrid=True, gridcolor="grey", dtick=60),
    )
)

fig.write_html("../outputs/fire_per_country.html")

In [100]:
# make a map with density of fires per country
fire_density_per_country = fire_per_country.copy() 
fire_density_per_country["density_100km2"] = fire_density_per_country["fire_count"] / (fire_density_per_country["AREA_KM2"]) *10000 # add clolumn density per 100km2

# add log column, log(0) is undefined so use log(x+1)
fire_density_per_country["density_log"] = np.log1p(fire_density_per_country["density_100km2"])

# inspect the distributen of the density
fire_density_per_country.describe()

,AREA_KM2,fire_count,density_100km2,density_log
count,2.580000e+02,258.000000,258.000000,258.000000
mean,3.423154e+07,509.616279,10.842608,1.098614
std,5.295922e+08,2006.668786,32.389615,1.366239
min,2.204709e-02,0.000000,0.000000,0.000000
25%,9.801617e+02,0.000000,0.000000,0.000000
50%,8.692411e+04,7.000000,0.776237,0.574495
75%,5.411109e+05,179.250000,5.229007,1.829199
max,8.507102e+09,21684.000000,284.900119,5.655643


In [107]:
# render map linar scale
geojson_density = json.loads(fire_density_per_country.to_json())

# color scale
colorscale = [
    [0, "lightgrey"],
    [0.0001, "lightyellow"],
    [0.5, "orange"],
    [1, "darkred"]
]

fig = go.Figure(go.Choropleth(
    geojson=geojson_density,
    locations=fire_density_per_country["NAME"],
    featureidkey="properties.NAME",
    z=fire_density_per_country["density_100km2"],
    colorscale=colorscale,
    colorbar_title="Fires per 100km²",
    customdata=fire_density_per_country[["NAME", "fire_count", "density_100km2"]].values,
    hovertemplate="<b>%{customdata[0]}</b><br>Fires recorded: %{customdata[1]}<br>Fires per 100km²: %{customdata[2]:.2f}<extra></extra>",
))

fig.update_layout(
    title=dict(text="Fire Density per 100km² by Country", x=0.5, xanchor="center", font=dict(size=24)),
    geo=dict(
        showframe=True,
        framecolor="grey",
        showland=True,
        landcolor="lightblue", # same color as ocean
        showcoastlines=False,
        showocean=True,
        oceancolor="lightblue",
        showlakes=True,
        lakecolor="lightblue",
        lataxis=dict(showgrid=True, gridcolor="grey", dtick=30),
        lonaxis=dict(showgrid=True, gridcolor="grey", dtick=60),
    )
)

fig.write_html("../outputs/fire_density_per_country_linear.html")

In [113]:
# render map logarithmic scale
geojson_density = json.loads(fire_density_per_country.to_json())

# color scale
colorscale = [
    [0, "lightgrey"],
    [0.0001, "lightyellow"],
    [0.5, "orange"],
    [1, "darkred"]
]

fig = go.Figure(go.Choropleth(
    geojson=geojson_density,
    locations=fire_density_per_country["NAME"],
    featureidkey="properties.NAME",
    z=fire_density_per_country["density_log"],
    colorscale=colorscale,
    colorbar_title="Fires per 100km²",
    customdata=fire_density_per_country[["NAME", "fire_count", "density_100km2"]].values,
    hovertemplate="<b>%{customdata[0]}</b><br>Fires recorded: %{customdata[1]}<br>Fires per 100km²: %{customdata[2]:.2f}<extra></extra>",
))

fig.update_layout(
    title=dict(text="Fire Density per 100km² by Country", x=0.5, xanchor="center", font=dict(size=24)),
    geo=dict(
        showframe=True,
        framecolor="grey",
        showland=True,
        landcolor="lightblue", # same color as ocean
        showcoastlines=False,
        showocean=True,
        oceancolor="lightblue",
        showlakes=True,
        lakecolor="lightblue",
        lataxis=dict(showgrid=True, gridcolor="grey", dtick=30),
        lonaxis=dict(showgrid=True, gridcolor="grey", dtick=60),
    )
)

fig.write_html("../outputs/fire_density_per_country_logarithmic.html")

In [137]:
# make a heatmap of the fires
fig = px.density_map(
    wildfire_gdf_crs4326,
    lat=wildfire_gdf_crs4326.geometry.y,
    lon=wildfire_gdf_crs4326.geometry.x,
    radius=5,
    zoom=1,
    map_style="carto-darkmatter",
    title="Wildfire Heatmap",
    color_continuous_scale="Hot",
    hover_data={"acq_date_day": True}
)

fig.update_traces(hovertemplate="Date: %{customdata[0]}<extra></extra>")

fig.update_layout(
    title=dict(text="Global Wildfire Heatmap", x=0.5, xanchor="center", font=dict(size=24)),
    coloraxis_showscale=False,
)

fig.write_html("../outputs/fire_heatmap.html")